In [ ]:
import requests
import pandas as pd
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("NASA_API_KEY")
BASE_URL = "https://api.nasa.gov/neo/rest/v1/neo/browse"


In [ ]:
CWD = Path.cwd().parent
DATASET = CWD / 'Datasets'

In [ ]:
columns = ['Absolute_Magnitude', 'Est_Dia_In_Km_Min', 'Est_Dia_In_Km_Max',
           'Close_Approach_Date', 'Relative_Velocity_Km_Per_Hr',
           'Miss_Dist_Kilometers', 'Minimum_Orbit_Intersection',
           'Jupiter_Tisserand_Invariant', 'Epoch_Osculation', 'Eccentricity',
           'Semi_Major_Axis', 'Inclination', 'Asc_Node_Longitude',
           'Orbital_Period', 'Perihelion_Distance', 'Perihelion_Arg',
           'Aphelion_Dist', 'Perihelion_Time', 'Mean_Anomaly', 'Mean_Motion',
           'Hazardous']

data_list = []  

In [ ]:
page = 0
while True:
    response = requests.get(BASE_URL, params={"api_key": API_KEY, "page": page})
    response.raise_for_status()
    data = response.json()
    
    neos = data['near_earth_objects']
    
    if not neos:
        break
    
    for neo in neos:
        # basic properties
        row = {
            'Absolute_Magnitude': neo.get('absolute_magnitude_h'),
            'Est_Dia_In_Km_Min': neo['estimated_diameter']['kilometers']['estimated_diameter_min'],
            'Est_Dia_In_Km_Max': neo['estimated_diameter']['kilometers']['estimated_diameter_max'],
            'Hazardous': neo.get('is_potentially_hazardous_asteroid')
        }

        # orbit data
        orb = neo['orbital_data']
        row.update({
            'Minimum_Orbit_Intersection': float(orb.get('minimum_orbit_intersection', 0)),
            'Jupiter_Tisserand_Invariant': float(orb.get('jupiter_tisserand_invariant', 0)),
            'Epoch_Osculation': float(orb.get('epoch_osculation', 0)),
            'Eccentricity': float(orb.get('eccentricity', 0)),
            'Semi_Major_Axis': float(orb.get('semi_major_axis', 0)),
            'Inclination': float(orb.get('inclination', 0)),
            'Asc_Node_Longitude': float(orb.get('ascending_node_longitude', 0)),
            'Orbital_Period': float(orb.get('orbital_period', 0)),
            'Perihelion_Distance': float(orb.get('perihelion_distance', 0)),
            'Perihelion_Arg': float(orb.get('perihelion_argument', 0)),
            'Aphelion_Dist': float(orb.get('aphelion_distance', 0)),
            'Perihelion_Time': float(orb.get('perihelion_time', 0)),
            'Mean_Anomaly': float(orb.get('mean_anomaly', 0)),
            'Mean_Motion': float(orb.get('mean_motion', 0))
        })

        if neo['close_approach_data']:
            cad = neo['close_approach_data'][0]
            row['Close_Approach_Date'] = cad['close_approach_date']
            row['Relative_Velocity_Km_Per_Hr'] = float(cad['relative_velocity']['kilometers_per_hour'])
            row['Miss_Dist_Kilometers'] = float(cad['miss_distance']['kilometers'])
        else:
            row['Close_Approach_Date'] = None
            row['Relative_Velocity_Km_Per_Hr'] = None
            row['Miss_Dist_Kilometers'] = None

        data_list.append(row)
    page += 1
    
    if len(data_list) >= 10000: # limit to 10000 records
        print('Reached 10000 records, stopping fetch.')
        break


In [ ]:
print('Total NEOs fetched from API:', len(data_list))

In [ ]:
output = DATASET / 'api_nasa_neo_data.csv'

if not output.exists(): output.touch()

df = pd.DataFrame(data_list, columns=columns)
df.to_csv(output, index=False)
print(f"Data saved to {output}")